1) IMPORT PUMP LIBRARY

In [5]:
import nesp_lib

2) CHECK PUMP PORT (USUALLY COM 5. RUN THIS IF NOT TO FIND OUT)

In [6]:
from nesp_lib import Port

# Constructs and opens the port to which the pump is connected.
port = Port("COM6", baud_rate=9600)

3) INITIALIZE PUMP (NO NEED)

In [7]:
import serial
ser = serial.Serial(
    port="COM6",
    baudrate=9600,
    stopbits=serial.STOPBITS_TWO,
    bytesize=serial.EIGHTBITS
 )

SerialException: could not open port 'COM6': PermissionError(13, 'Access is denied.', None, 5)

4) PUMP SETUP CODE

In [8]:
from nesp_lib import Pump, PumpingDirection

# Constructs the pump connected to the port.
pump = Pump(port)

# Sets the syringe diameter of the pump in units of millimeters.
pump.syringe_diameter = 26.67 # 26.67 mm can be changed

# Sets the pumping direction of the pump.
pump.pumping_direction = PumpingDirection.INFUSE

# Sets the pumping volume of the pump in units of milliliters.
pump.pumping_volume = 30.0

# Sets the pumping rate of the pump in units of milliliters per minute.
pump.pumping_rate = 1 # SYRINGE PUMP

# Prints the model number of the pump (e.g. "1000" for NE-1000).
print(pump.model_number)

# Prints the firmware version of the pump (e.g. "(3, 928)" for 3.928).
print(pump.firmware_version)

1000
(3, 934)


5) PUMP TEST RUN BELOW (OPTIONAL)

In [32]:
# Runs the pump considering the direction, volume, and rate set.
# Sets the pumping volume of the pump in units of milliliters.
pump.pumping_volume = 30.0

# Sets the pumping rate of the pump in units of milliliters per minute.
pump.pumping_rate = 1 # SYRINGE PUMP
pump.run()

6) EXPERIMENT INITIALIZATION

In [38]:
# --- One-Time User Input Script ---

pig_weight = float(input("Enter pig weight (kg): "))
drug_name = input("Enter drug name: ")
drug_concentration = input("Enter drug infusion concentration (e.g., 1 mg/mL): ")

print("\nINPUT SUMMARY:")
print(f"Pig Weight: {pig_weight} kg")
print(f"Drug: {drug_name}")
print(f"Infusion Concentration: {drug_concentration}")

# Build filename: PIG_80kg_Epinephrine_16.csv
output_filename = f"PIG_{pig_weight}kg_{drug_name}_{drug_concentration}.csv"

print(f"\nOutput file will be: {output_filename}\n")



INPUT SUMMARY:
Pig Weight: 40.2 kg
Drug: norepi
Infusion Concentration: 60

Output file will be: PIG_40.2kg_norepi_60.csv



7) PID TEST CODE BELOW for basic ACHIEVE MAP

In [40]:
import time
import pandas as pd
from simple_pid import PID

# -------------------------------
# Read MAP from CSV
# -------------------------------
def read_current_row(file_path):
    df = pd.read_csv(file_path, header=None)
    last_row = df.iloc[-1]               # full row
    current_MAP = float(last_row[3])     # MAP = column 3
    return current_MAP, last_row


def read_current_MAP(file_path):
    df = pd.read_csv(file_path, header=None)
    last_row = df.iloc[-1]

    current_MAP = int(last_row[3])   # MAP = column 3 (your number3)

    return current_MAP


# -------------------------------
# PID Setup
# -------------------------------
pid = PID(0.001, 0.0005, 0.00005, setpoint=40)   # PID terms + target MAP, FOR EPI keep positive. FOR Clavidipine make negative.
pid.sample_time = 5                      # PID updates every 5 seconds
#pid.output_limits = (0, 200)             # optional safety limits
pid.output_limits = (0, 27) 

# -------------------------------
# MAIN LOOP
# -------------------------------
csv_file_path = '../raghav code/DashExportData.csv'

initial_rows = len(pd.read_csv(csv_file_path, header=None))
print(f"Waiting for new MAP row... (current rows = {initial_rows})")

# STEP 2 — WAIT for ONE new row ONLY
while True:
    df = pd.read_csv(csv_file_path, header=None)
    current_rows = len(df)

    if current_rows > initial_rows:
        print("New MAP row detected — starting control loop.")
        break

    time.sleep(0.001)   # short wait to avoid busy CPU spinning

# -------------------------------
# 5. CONTINUOUS PID LOOP
# -------------------------------
# Prepare CSV header
header_written = False

while True:

    # 1. Read MAP + complete raw row
    current_MAP, last_row = read_current_row(csv_file_path)

    # 2. PID output (infusion rate in mL/min)
    control = pid(current_MAP)

    # 3. Pump control
    pump.pumping_rate = control          # mL/min
    pump.pumping_volume = 4.9*(control/60) # mL delivered over 5 sec

   # total_volume=total_volume + pump.pumping_volume

    # 4. Convert infusion rate to mcg/kg/min
    #    control = mL/min
    #    drug_concentration = mcg/mL
    #    pig_weight = kg
   # dose_mcg_per_kg_min = (control * drug_concentration) / pig_weight
    #print(f"MAP={current_MAP} | Rate={control:.3f} mL/min | Dose={dose_mcg_per_kg_min:.3f} mcg/kg/min\n")
    pump.run()
    # 5. Build output row
    output_row = list(last_row.values)   # all MAP CSV columns
    output_row.append(control)           # add infusion rate
    #output_row.append(dose_mcg_per_kg_min)  # add mcg/kg/min dose

    # 6. Write to CSV
    if not header_written:
        # Use the exact headers from the MAP CSV
        original_headers = list(last_row.index)  
        # Append new columns for infusion data
        new_headers = original_headers + ["infusion_rate_mL_min", "dose_mcg_kg_min"]
    
        # Create CSV with correct header row
        pd.DataFrame(columns=new_headers).to_csv(output_filename, index=False)
        header_written = True

    # Append row
    pd.DataFrame([output_row]).to_csv(output_filename, mode='a', header=False, index=False)



Waiting for new MAP row... (current rows = 963)
New MAP row detected — starting control loop.


KeyboardInterrupt: 

8) PUMP RESET

In [39]:
# Sets the pumping direction of the pump.
pump.pumping_direction = PumpingDirection.WITHDRAW
pump.pumping_rate = 28.0 
pump.pumping_volume=total_volume
pump.run()


StatusAlarmException: AlarmStatus.STALLED